In [1]:
"""
German "holiday fingerprint" matrix built with the `holidays` package.

The `holidays` package computes public holidays algorithmically for any year
and covers all 16 German federal states, so it works for the Rossmann range
(2013-2015). The fingerprint is a daily (date x state) matrix flagging, for
every state, whether each day is a public holiday -- mirroring the Rossmann
`StateHoliday` column. (School holidays are NOT derivable here; Rossmann
already ships its own `SchoolHoliday` column.)
"""

import holidays
import pandas as pd

COUNTRY = "DE"
LANGUAGE = "en_US"  # language for holiday names returned by the package

# Rossmann training data spans 2013-01-01 .. 2015-07-31; pull whole years.
YEARS = [2013, 2014, 2015]
VALID_FROM = "2013-01-01"
VALID_TO = "2015-12-31"

print("holidays package version:", holidays.__version__)


holidays package version: 0.99


In [2]:
# ISO 3166-2 subdivision code -> federal state name.
STATE_NAMES = {
    "BW": "Baden-Württemberg", "BY": "Bavaria", "BE": "Berlin",
    "BB": "Brandenburg", "HB": "Bremen", "HH": "Hamburg",
    "HE": "Hesse", "MV": "Mecklenburg-Western Pomerania", "NI": "Lower Saxony",
    "NW": "North Rhine-Westphalia", "RP": "Rhineland-Palatinate",
    "SL": "Saarland", "SN": "Saxony", "ST": "Saxony-Anhalt",
    "SH": "Schleswig-Holstein", "TH": "Thuringia",
}

# Pull supported subdivisions from the package and keep only the 16 federal
# states (the package also exposes city-level entries such as "Augsburg").
supported = set(holidays.Germany(years=YEARS).subdivisions)
state_codes_all = [c for c in STATE_NAMES if c in supported]

subdivisions = pd.DataFrame(
    [{"subdiv": code, "isoCode": f"DE-{code}", "name": STATE_NAMES[code]}
     for code in state_codes_all]
).sort_values("subdiv").reset_index(drop=True)

print(f"Found {len(subdivisions)} German federal states:")
subdivisions


Found 16 German federal states:


,subdiv,isoCode,name
0,BB,DE-BB,Brandenburg
1,BE,DE-BE,Berlin
2,BW,DE-BW,Baden-Württemberg
3,BY,DE-BY,Bavaria
4,HB,DE-HB,Bremen
5,HE,DE-HE,Hesse
6,HH,DE-HH,Hamburg
7,MV,DE-MV,Mecklenburg-Western Pomerania
8,NI,DE-NI,Lower Saxony
9,NW,DE-NW,North Rhine-Westphalia


In [3]:
# ---------------------------------------------------------------------------
# 2. Extract public holidays for a given state and set of years
# ---------------------------------------------------------------------------

def get_public_holidays(subdiv: str, years=YEARS, language: str = LANGUAGE) -> pd.DataFrame:
    """Return one row per public holiday for a German state across `years`."""
    cal = holidays.country_holidays(
        COUNTRY, subdiv=subdiv, years=years, language=language
    )
    rows = [
        {"Date": pd.Timestamp(d), "name": name, "subdiv": subdiv}
        for d, name in sorted(cal.items())
    ]
    return pd.DataFrame(rows, columns=["Date", "name", "subdiv"])


# Quick sanity check for a single state (Bavaria) -- should now be non-empty.
bavaria_public = get_public_holidays("BY")
print(f"Bavaria public holidays {VALID_FROM}..{VALID_TO}: {len(bavaria_public)} entries")
bavaria_public.head(15)


Bavaria public holidays 2013-01-01..2015-12-31: 36 entries


,Date,name,subdiv
0,2013-01-01,New Year's Day,BY
1,2013-01-06,Epiphany,BY
2,2013-03-29,Good Friday,BY
3,2013-04-01,Easter Monday,BY
4,2013-05-01,Labor Day,BY
5,2013-05-09,Ascension Day,BY
6,2013-05-20,Whit Monday,BY
7,2013-05-30,Corpus Christi,BY
8,2013-10-03,German Unity Day,BY
9,2013-11-01,All Saints' Day,BY


In [4]:
# ---------------------------------------------------------------------------
# 3. Loop over all states and collect every public holiday (long form)
# ---------------------------------------------------------------------------

def collect_all(codes, years=YEARS) -> pd.DataFrame:
    """Fetch public holidays for every subdivision and concatenate the results."""
    frames = []
    for code in codes:
        df = get_public_holidays(code, years=years)
        print(f"  PublicHolidays {code}: {len(df)} entries")
        frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


state_codes = subdivisions["subdiv"].tolist()

print("Collecting public holidays for all states...")
public_holidays = collect_all(state_codes)

print(f"\nTotal public holiday rows: {len(public_holidays)}")
print(f"Unique holiday names: {public_holidays['name'].nunique()}")
public_holidays.head()


  PublicHolidays BB: 36 entries
  PublicHolidays BE: 27 entries
  PublicHolidays BW: 36 entries
  PublicHolidays BY: 36 entries
  PublicHolidays HB: 27 entries
  PublicHolidays HE: 30 entries
  PublicHolidays HH: 27 entries
  PublicHolidays MV: 30 entries
  PublicHolidays NI: 27 entries
  PublicHolidays NW: 33 entries
  PublicHolidays RP: 33 entries
  PublicHolidays SH: 27 entries
  PublicHolidays SL: 36 entries
  PublicHolidays SN: 33 entries
  PublicHolidays ST: 33 entries
  PublicHolidays TH: 30 entries

Total public holiday rows: 501
Unique holiday names: 17


,Date,name,subdiv
0,2013-01-01,New Year's Day,BB
1,2013-03-29,Good Friday,BB
2,2013-03-31,Easter Sunday,BB
3,2013-04-01,Easter Monday,BB
4,2013-05-01,Labor Day,BB


In [5]:
# ---------------------------------------------------------------------------
# 4. Build the daily "holiday fingerprint" matrix (date x state)
# ---------------------------------------------------------------------------

def build_fingerprint(public_df: pd.DataFrame, codes,
                      start: str = VALID_FROM, end: str = VALID_TO) -> pd.DataFrame:
    """Return a daily matrix with a PublicHoliday flag for each state."""
    flags = (public_df
             .assign(PublicHoliday=1)
             .drop_duplicates(subset=["Date", "subdiv"])
             [["Date", "subdiv", "PublicHoliday"]])

    # Full grid of every calendar day x every state, then left-join the flags.
    all_days = pd.date_range(start, end, freq="D")
    grid = pd.MultiIndex.from_product(
        [all_days, codes], names=["Date", "subdiv"]
    ).to_frame(index=False)

    fp = grid.merge(flags, on=["Date", "subdiv"], how="left")
    fp["PublicHoliday"] = fp["PublicHoliday"].fillna(0).astype(int)
    return fp.sort_values(["Date", "subdiv"]).reset_index(drop=True)


fingerprint = build_fingerprint(public_holidays, state_codes)
print(f"Fingerprint shape: {fingerprint.shape}")
print(f"Public-holiday day-state pairs: {int(fingerprint['PublicHoliday'].sum())}")
fingerprint.head(20)


Fingerprint shape: (17520, 3)
Public-holiday day-state pairs: 501


,Date,subdiv,PublicHoliday
0,2013-01-01,BB,1
1,2013-01-01,BE,1
2,2013-01-01,BW,1
3,2013-01-01,BY,1
4,2013-01-01,HB,1
5,2013-01-01,HE,1
6,2013-01-01,HH,1
7,2013-01-01,MV,1
8,2013-01-01,NI,1
9,2013-01-01,NW,1


In [6]:
# ---------------------------------------------------------------------------
# 5. Wide fingerprint (one column per state) + save to disk
# ---------------------------------------------------------------------------

# Wide layout: each state becomes a column, handy as a per-day feature vector.
fingerprint_wide = (fingerprint
                    .pivot(index="Date", columns="subdiv", values="PublicHoliday")
                    .add_prefix("pub_")
                    .reset_index())

# A nationwide count: how many of the 16 states observe a public holiday per day.
fingerprint_wide["n_states_public_holiday"] = (
    fingerprint_wide.filter(like="pub_").sum(axis=1)
)

# Persist both the long and wide forms for downstream feature engineering.
fingerprint.to_csv("holiday_fingerprint_long.csv", index=False)
fingerprint_wide.to_csv("holiday_fingerprint_wide.csv", index=False)

print(f"Saved long form : holiday_fingerprint_long.csv {fingerprint.shape}")
print(f"Saved wide form : holiday_fingerprint_wide.csv {fingerprint_wide.shape}")
fingerprint_wide.head()


Saved long form : holiday_fingerprint_long.csv (17520, 3)
Saved wide form : holiday_fingerprint_wide.csv (1095, 18)


subdiv,Date,pub_BB,pub_BE,pub_BW,pub_BY,pub_HB,pub_HE,pub_HH,pub_MV,pub_NI,pub_NW,pub_RP,pub_SH,pub_SL,pub_SN,pub_ST,pub_TH,n_states_public_holiday
0,2013-01-01,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,16
1,2013-01-02,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,2013-01-03,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,2013-01-04,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,2013-01-05,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [7]:
import pandas as pd 

""" Read the Rossmann store sales dataset and preprocess it for modeling. """
from src.preprocessing import preprocess_data, interpolate_closed_days, set_closed_to_zero


sales = pd.read_csv(r"C:/Users/Miltos.KALIKATZAR/Downloads/stuff/datasets/rossmann-store-sales/train.csv")
stores = pd.read_csv(r"C:/Users/Miltos.KALIKATZAR/Downloads/stuff/datasets/rossmann-store-sales/store.csv")

df = preprocess_data(sales, stores)


C:\Users\Miltos.KALIKATZAR\AppData\Local\Temp\ipykernel_15672\685619000.py:7: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  sales = pd.read_csv(r"C:/Users/Miltos.KALIKATZAR/Downloads/stuff/datasets/rossmann-store-sales/train.csv")


In [8]:
df.columns

Index(['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo',
       'StoreType', 'Assortment', 'CompetitionDistance', 'Promo2',
       'PromoInterval', 'Promo2SinceDate', 'CompetitionStartDate',
       'isStateHoliday', 'isSchoolHoliday'],
      dtype='object')

In [9]:
holiday_df= df[['Date', 'Store', 'isStateHoliday', 'isSchoolHoliday']]

In [10]:
holiday_df

,Date,Store,isStateHoliday,isSchoolHoliday
1016095,2013-01-01,1,True,True
1014980,2013-01-02,1,False,True
1013865,2013-01-03,1,False,True
1012750,2013-01-04,1,False,True
1011635,2013-01-05,1,False,True
...,...,...,...,...
5574,2015-07-27,1115,False,True
4459,2015-07-28,1115,False,True
3344,2015-07-29,1115,False,True
2229,2015-07-30,1115,False,True


In [11]:
# ---------------------------------------------------------------------------
# 6. Infer each store's federal state from its StateHoliday pattern
# ---------------------------------------------------------------------------
# Idea: every store sits in exactly one state, so the dates on which it reports
# `isStateHoliday == True` should match that state's public-holiday fingerprint.
# Regional holidays (Epiphany, Corpus Christi, Reformation Day, ...) differ
# between states, which is what makes a store's pattern identifiable.
#
# Scoring: Hamming agreement. For every (store, candidate-state) pair we count
# the days where the store's observed flag equals the state's fingerprint flag,
# then pick the state with the highest agreement. Fully vectorised with NumPy
# so it scales to ~1M rows x 16 states without Python-level loops.
#
# Caveat: some states shared an IDENTICAL public-holiday calendar in 2013-2015
# (e.g. BE/HB/HH/NI/SH = national-only; NW/RP; MV/TH). Those are mathematically
# indistinguishable from public holidays alone, so we report the full tied set.

import numpy as np

state_cols = [c for c in fingerprint_wide.columns if c.startswith("pub_")]
state_of_col = {c: c[len("pub_"):] for c in state_cols}
codes = np.array([state_of_col[c] for c in state_cols])

# Align dates, then attach the 16 per-state public-holiday flags to every row.
hd = holiday_df[["Store", "Date", "isStateHoliday"]].copy()
hd["Date"] = pd.to_datetime(hd["Date"]).dt.normalize()

merged = hd.merge(fingerprint_wide[["Date"] + state_cols], on="Date", how="inner")

obs = merged["isStateHoliday"].to_numpy().astype(np.int8)        # (N,)   observed
M = merged[state_cols].to_numpy().astype(np.int8)                # (N,16) candidates
store_ids = merged["Store"].to_numpy()

# Per-row agreement with each state (1 = match), then sum per store.
agree = (M == obs[:, None]).astype(np.int32)                     # (N,16)
scores = (pd.DataFrame(agree, columns=state_cols)
          .groupby(store_ids, sort=True).sum())
scores.index.name = "Store"
n_obs = (pd.Series(np.ones(len(store_ids), np.int64))
         .groupby(store_ids, sort=True).sum().to_numpy())

# Best state per store + all states tied at the top (the ambiguity set).
score_mat = scores.to_numpy()
best_score = score_mat.max(axis=1)
ties = score_mat == best_score[:, None]                          # (S,16) bool
best_idx = score_mat.argmax(axis=1)                              # representative pick
second_score = np.sort(score_mat, axis=1)[:, -2]

store_state = pd.DataFrame({
    "Store": scores.index,
    "state": codes[best_idx],                                    # representative state
    "candidate_states": [",".join(codes[row]) for row in ties],
    "n_candidates": ties.sum(axis=1),
    "match_frac": best_score / n_obs,                            # 1.0 = perfect fit
    "margin": (best_score - second_score) / n_obs,              # 0 => tie present
})
store_state["state_name"] = store_state["state"].map(STATE_NAMES)
store_state["ambiguous"] = store_state["n_candidates"] > 1

print(f"Inferred state for {len(store_state)} stores "
      f"(mean match fraction {store_state['match_frac'].mean():.4f}).")
print(f"Uniquely identified : {(~store_state['ambiguous']).sum()}")
print(f"Ambiguous (tied set): {store_state['ambiguous'].sum()}")
print("\nDistinct tied candidate groups (states with identical calendars):")
print(store_state.loc[store_state['ambiguous'], 'candidate_states'].value_counts())
store_state.head()


Inferred state for 1115 stores (mean match fraction 0.9995).
Uniquely identified : 243
Ambiguous (tied set): 872

Distinct tied candidate groups (states with identical calendars):
candidate_states
NW,RP             326
BE,HB,HH,NI,SH    257
BW,BY             253
MV,TH              36
Name: count, dtype: int64


,Store,state,candidate_states,n_candidates,match_frac,margin,state_name,ambiguous
0,1,HE,HE,1,1.000000,0.002123,Hesse,False
1,2,MV,"MV,TH",2,0.998938,0.000000,Mecklenburg-Western Pomerania,True
2,3,NW,"NW,RP",2,1.000000,0.000000,North Rhine-Westphalia,True
3,4,BE,"BE,HB,HH,NI,SH",5,1.000000,0.000000,Berlin,True
4,5,SN,SN,1,0.996815,0.001062,Saxony,False


In [12]:
# ---------------------------------------------------------------------------
# 7. Attach the inferred state back to every row
# ---------------------------------------------------------------------------
# `state` is the representative pick; `candidate_states` keeps the full tied set
# so downstream code can decide how to treat ambiguous stores. The map-based
# join is O(rows) and stays cheap on the full ~1M-row frame.

store_to_state = store_state.set_index("Store")
holiday_df = holiday_df.copy()
holiday_df["state"] = holiday_df["Store"].map(store_to_state["state"])
holiday_df["state_name"] = holiday_df["Store"].map(store_to_state["state_name"])
holiday_df["candidate_states"] = holiday_df["Store"].map(store_to_state["candidate_states"])
holiday_df["state_ambiguous"] = holiday_df["Store"].map(store_to_state["ambiguous"])

assert holiday_df["state"].notna().all(), "Some rows have no inferred state"
print(f"Annotated {len(holiday_df):,} rows with a store-level state.")
holiday_df.head()


Annotated 1,017,209 rows with a store-level state.


,Date,Store,isStateHoliday,isSchoolHoliday,state,state_name,candidate_states,state_ambiguous
1016095,2013-01-01,1,True,True,HE,Hesse,HE,False
1014980,2013-01-02,1,False,True,HE,Hesse,HE,False
1013865,2013-01-03,1,False,True,HE,Hesse,HE,False
1012750,2013-01-04,1,False,True,HE,Hesse,HE,False
1011635,2013-01-05,1,False,True,HE,Hesse,HE,False


In [13]:
# ---------------------------------------------------------------------------
# 8. Attach the public-holiday NAME for each (store, date)
# ---------------------------------------------------------------------------
# A holiday's name depends only on (Date, state), so we build a tiny lookup of
# at most (#days x 16 states) rows and merge it onto the full ~1M-row frame in
# one vectorised pass -- O(rows) hash join, no Python-level loops. Each store's
# inferred `state` selects which state's calendar applies. (Ambiguous stores
# share an identical calendar across their tied states, so the representative
# `state` pick yields the same name.)

# (Date, state) -> holiday name. Some days carry >1 holiday in a state, so
# concatenate the distinct names (order-preserving) instead of dropping them.
holiday_names = (
    public_holidays
    .assign(Date=pd.to_datetime(public_holidays["Date"]).dt.normalize())
    .groupby(["Date", "subdiv"], sort=False)["name"]
    .agg(lambda s: " / ".join(dict.fromkeys(s)))
    .reset_index()
    .rename(columns={"subdiv": "state", "name": "holiday_name", "Date": "_Date_norm"})
)

holiday_df = holiday_df.copy()
holiday_df["_Date_norm"] = pd.to_datetime(holiday_df["Date"]).dt.normalize()
holiday_df = holiday_df.merge(holiday_names, on=["_Date_norm", "state"], how="left")
holiday_df = holiday_df.drop(columns="_Date_norm")

# Non-holiday rows get a NaN name -> normalise to empty string for a clean CSV.
holiday_df["holiday_name"] = holiday_df["holiday_name"].fillna("")

n_named = (holiday_df["holiday_name"] != "").sum()
print(f"Tagged {n_named:,} of {len(holiday_df):,} rows with a public-holiday name "
      f"({n_named / len(holiday_df):.2%}).")
print("\nTop holiday names by row count:")
print(holiday_df.loc[holiday_df["holiday_name"] != "", "holiday_name"].value_counts().head(10))

holiday_df.to_csv("rossmann_holiday_with_state.csv", index=False)
holiday_df.head()


Tagged 30,681 of 1,017,209 rows with a public-holiday name (3.02%).

Top holiday names by row count:
holiday_name
Good Friday                3345
Easter Monday              3345
Ascension Day              3345
Labor Day                  3345
Whit Monday                3345
New Year's Day             3344
Corpus Christi             2073
German Unity Day           2050
Christmas Day              2050
Second Day of Christmas    2050
Name: count, dtype: int64


,Date,Store,isStateHoliday,isSchoolHoliday,state,state_name,candidate_states,state_ambiguous,holiday_name
0,2013-01-01,1,True,True,HE,Hesse,HE,False,New Year's Day
1,2013-01-02,1,False,True,HE,Hesse,HE,False,
2,2013-01-03,1,False,True,HE,Hesse,HE,False,
3,2013-01-04,1,False,True,HE,Hesse,HE,False,
4,2013-01-05,1,False,True,HE,Hesse,HE,False,
